In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_squared_error

print("Libraries loaded successfully!")

# Locate competition data
train_files = list(Path("/kaggle/input").rglob("train.csv"))

if not train_files:
    raise FileNotFoundError(
        "train.csv not found. Please check Add Input."
    )

test_files = list(Path("/kaggle/input").rglob("test.csv"))

if not test_files:
    raise FileNotFoundError(
        "test.csv not found. Please check Add Input."
    )

train = pd.read_csv(train_files[0])
test = pd.read_csv(test_files[0])

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Libraries loaded successfully!
Train shape: (1460, 81)
Test shape: (1459, 80)


In [2]:
def add_features(df):
    df = df.copy()

    df["TotalSF"] = (
        df["TotalBsmtSF"].fillna(0)
        + df["1stFlrSF"].fillna(0)
        + df["2ndFlrSF"].fillna(0)
    )

    df["TotalBathrooms"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )

    porch_cols = [
        "OpenPorchSF",
        "3SsnPorch",
        "EnclosedPorch",
        "ScreenPorch",
        "WoodDeckSF"
    ]

    df["TotalPorchSF"] = (
        df[porch_cols].fillna(0).sum(axis=1)
    )

    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]

    df["OverallQual_SF"] = (
        df["OverallQual"].fillna(0)
        * df["GrLivArea"].fillna(0)
    )

    return df


train = add_features(train)
test = add_features(test)

print("Feature engineering completed.")
print("Train:", train.shape)
print("Test:", test.shape)

Feature engineering completed.
Train: (1460, 87)
Test: (1459, 86)


In [3]:
X = train.drop(
    columns=["SalePrice", "Id"],
    errors="ignore"
)

y = np.log1p(train["SalePrice"])

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

Training rows: 1168
Validation rows: 292


In [4]:
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline ready.")

Preprocessing pipeline ready.


In [5]:
models = {
    "Dummy Baseline": DummyRegressor(strategy="mean"),

    "Ridge Regression": Ridge(alpha=10),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=3,
        min_samples_leaf=3,
        random_state=42
    )
}

results = []

for name, model in models.items():
    print(f"Training {name}...")

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", clone(model))
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    results.append({
        "Model": name,
        "Validation_Log_RMSE": rmse
    })

    print(f"{name} RMSE: {rmse:.5f}")

results_df = pd.DataFrame(results).sort_values(
    "Validation_Log_RMSE"
).reset_index(drop=True)

print("\nModel comparison:")
display(results_df)

Training Dummy Baseline...
Dummy Baseline RMSE: 0.43324
Training Ridge Regression...
Ridge Regression RMSE: 0.13470
Training Random Forest...
Random Forest RMSE: 0.14523
Training Gradient Boosting...
Gradient Boosting RMSE: 0.14055

Model comparison:


,Model,Validation_Log_RMSE
0,Ridge Regression,0.134697
1,Gradient Boosting,0.140550
2,Random Forest,0.145227
3,Dummy Baseline,0.433244


In [6]:
results_df.to_csv(
    "/kaggle/working/model_results.csv",
    index=False
)

print("Saved: /kaggle/working/model_results.csv")
display(results_df)

Saved: /kaggle/working/model_results.csv


,Model,Validation_Log_RMSE
0,Ridge Regression,0.134697
1,Gradient Boosting,0.140550
2,Random Forest,0.145227
3,Dummy Baseline,0.433244


In [7]:
# Step 17: Error Analysis using Ridge Regression

best_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(alpha=10))
    ]
)

# Train on the training split
best_pipeline.fit(X_train, y_train)

# Predict validation prices on the log scale
valid_pred_log = best_pipeline.predict(X_valid)

# Build an error analysis table
error_df = pd.DataFrame({
    "Id": train.loc[y_valid.index, "Id"].to_numpy(),
    "ActualSalePrice": np.expm1(y_valid.to_numpy()),
    "PredictedSalePrice": np.expm1(valid_pred_log),
    "AbsoluteLogError": np.abs(
        y_valid.to_numpy() - valid_pred_log
    ),
    "GrLivArea": train.loc[
        y_valid.index, "GrLivArea"
    ].to_numpy(),
    "OverallQual": train.loc[
        y_valid.index, "OverallQual"
    ].to_numpy()
})

# Show the 10 largest prediction errors
error_df = error_df.sort_values(
    "AbsoluteLogError",
    ascending=False
)

display(error_df.head(10))

# Save the error analysis
error_df.to_csv(
    "/kaggle/working/validation_error_analysis.csv",
    index=False
)

print("Error analysis completed and saved.")

,Id,ActualSalePrice,PredictedSalePrice,AbsoluteLogError,GrLivArea,OverallQual
197,31,40000.0,84887.379335,0.752433,1317,4
181,813,55993.0,99813.528903,0.578069,1044,5
134,917,35311.0,60820.402339,0.543719,480,2
227,1433,64500.0,105805.613611,0.494932,968,4
278,589,143000.0,230840.852018,0.478881,1473,5
109,629,135000.0,214378.722483,0.462467,2080,5
223,875,66500.0,97344.581025,0.381050,1131,5
27,278,141000.0,100566.006036,0.337943,864,4
74,899,611657.0,440102.617088,0.329163,2364,9
202,366,147000.0,106665.133137,0.320736,1344,5


Error analysis completed and saved.
